# Stage C v1 supervised instruction fine-tuning

Response-only full-parameter SFT from promoted Stage B v2 `checkpoint_00008000`. This notebook keeps the sealed SFT test split out of selection. Run the gates in order before starting either learning-rate pilot.

## Required Google Drive inputs

Upload `stage-c-sft-data.tar` and its `.sha256` sidecar to `MyDrive/medical-slm/`. Preserve the promoted Stage B v2 checkpoint under `MyDrive/medical-slm-runs/stage_b_v2/`. Create the archive locally with:

```powershell
python scripts/artifacts/create_stage_c_data_archive.py
```

## Bootstrap repository, Stage C data, tokenizer, and parent checkpoint

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY_BRANCH = 'main'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-c-sft-data.tar')
CHECKSUM_FILE = Path(str(DATA_ARCHIVE) + '.sha256')
DRIVE_PARENT_ROOT = Path('/content/drive/MyDrive/medical-slm-runs/stage_b_v2')
LOCAL_PARENT = Path('/content/stage_b_v2_parent/checkpoint_00008000')
DRIVE_ROOT = Path('/content/drive/MyDrive/medical-slm-runs/stage_c_sft_v1')
assert DATA_ARCHIVE.is_file() and CHECKSUM_FILE.is_file()
expected_hash = CHECKSUM_FILE.read_text().split()[0]
digest = hashlib.sha256()
with DATA_ARCHIVE.open('rb') as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == expected_hash
parents = sorted(DRIVE_PARENT_ROOT.rglob('checkpoint_00008000/checkpoint_manifest.json'))
assert parents, 'Promoted Stage B v2 checkpoint_00008000 was not found in Drive.'
DRIVE_PARENT = parents[0].parent
if REPOSITORY.exists() and not (REPOSITORY / '.git').is_dir():
    shutil.rmtree(REPOSITORY)
if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only', 'origin', REPOSITORY_BRANCH], check=True)
os.chdir(REPOSITORY)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
source_path = str(REPOSITORY / 'src')
if source_path not in sys.path: sys.path.insert(0, source_path)
importlib.invalidate_caches()
if not Path('datasets/tokenized/sft_stage_c_v1/manifest.json').is_file():
    subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
if not (LOCAL_PARENT / 'checkpoint_manifest.json').is_file():
    subprocess.run(['rsync', '-a', f'{DRIVE_PARENT}/', f'{LOCAL_PARENT}/'], check=True)
from medical_slm.training.checkpoint import verify_checkpoint
verify_checkpoint(LOCAL_PARENT)
manifest = json.loads(Path('datasets/tokenized/sft_stage_c_v1/manifest.json').read_text())
assert manifest['splits']['train']['examples'] == 6248
assert manifest['splits']['validation']['supervised_tokens'] == 68694
print({'gpu_ready': True, 'archive_sha256': expected_hash, 'parent': LOCAL_PARENT.name})

## Verify GPU, precision, focused regressions, and parent initialization

In [ ]:
import torch
from medical_slm.training.precision import resolve_precision
assert torch.cuda.is_available(), 'Select a GPU runtime.'
policy = resolve_precision('auto', 'cuda')
print({'gpu': torch.cuda.get_device_name(0), 'precision': policy.name})
subprocess.run(['python', '-m', 'pytest', 'tests/test_causal_loss.py', 'tests/test_sft_training.py', 'tests/test_training_config.py', 'tests/test_training_checkpoint.py', '-q'], check=True)
subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--parent-checkpoint', str(LOCAL_PARENT), '--verify-initialization-only'], check=True)
print('STAGE C INITIALIZATION GATE: PASSED')

## Zero-update SFT, medical, and general baseline

In [ ]:
BASELINE_LOCAL = Path('/content/stage_c_baseline.json')
BASELINE_DRIVE = DRIVE_ROOT / 'stage_c_baseline.json'
subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--parent-checkpoint', str(LOCAL_PARENT), '--baseline-only', '--baseline-output', str(BASELINE_LOCAL)], check=True)
baseline = json.loads(BASELINE_LOCAL.read_text())
assert baseline['optimizer_updates'] == 0
assert baseline['sft_validation']['tokens'] == 68694
assert baseline['medical_retention_validation']['tokens'] == 997632
assert baseline['general_retention_validation']['tokens'] == 466432
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(BASELINE_LOCAL, BASELINE_DRIVE)
print('STAGE C BASELINE GATE: PASSED', baseline)

## Ten-update one-batch response-mask alignment gate

In [ ]:
import yaml
OVERFIT_OUTPUT = Path('/content/stage_c_overfit')
if OVERFIT_OUTPUT.exists(): shutil.rmtree(OVERFIT_OUTPUT)
values = yaml.safe_load(Path('configs/training_stage_c_sft.yaml').read_text())
values.update({'output_directory': str(OVERFIT_OUTPUT), 'checkpoint_backup_directory': None, 'parent_checkpoint_directory': str(LOCAL_PARENT), 'precision': 'auto', 'max_updates': 10, 'log_interval': 1})
overfit_config = Path('/content/training_stage_c_overfit.yaml')
overfit_config.write_text(yaml.safe_dump(values, sort_keys=False))
subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--config', str(overfit_config), '--overfit-one-batch', '--max-updates', '10'], check=True)
records = [json.loads(line) for line in (OVERFIT_OUTPUT / 'metrics.jsonl').read_text().splitlines()]
losses = [row['metrics']['loss'] for row in records if row['event'] == 'overfit_one_batch']
assert len(losses) == 10 and losses[-1] < losses[0]
print('ONE-BATCH ALIGNMENT GATE: PASSED', losses[0], '->', losses[-1])

## Matched learning-rate pilot helpers

In [ ]:
from medical_slm.training.metrics import mirror_metric_log
PILOT_LRS = {'lr_1e5': 1e-5, 'lr_2e5': 2e-5}
def pilot_paths(arm):
    return Path(f'/content/stage_c_pilots/{arm}'), DRIVE_ROOT / 'pilots' / arm
def write_pilot_config(arm):
    local, drive_output = pilot_paths(arm)
    values = yaml.safe_load(Path('configs/training_stage_c_sft.yaml').read_text())
    values.update({'output_directory': str(local), 'checkpoint_backup_directory': str(drive_output / 'checkpoints'), 'parent_checkpoint_directory': str(LOCAL_PARENT), 'precision': 'auto', 'learning_rate': PILOT_LRS[arm], 'final_learning_rate': PILOT_LRS[arm] / 10, 'max_updates': 150})
    path = Path(f'/content/training_stage_c_{arm}.yaml')
    path.write_text(yaml.safe_dump(values, sort_keys=False))
    return path
def preserve_metrics(arm):
    local, drive_output = pilot_paths(arm)
    drive_output.mkdir(parents=True, exist_ok=True)
    if (local / 'metrics.jsonl').is_file(): mirror_metric_log(local / 'metrics.jsonl', drive_output / 'metrics.jsonl')
def run_fresh_pilot(arm):
    local, drive_output = pilot_paths(arm)
    assert not (drive_output / 'checkpoints/latest.json').exists(), 'Pilot exists; use resume_pilot.'
    if local.exists(): shutil.rmtree(local)
    subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--config', str(write_pilot_config(arm))], check=True)
    preserve_metrics(arm)
def resume_pilot(arm):
    local, drive_output = pilot_paths(arm)
    assert (drive_output / 'checkpoints/latest.json').is_file()
    (local / 'checkpoints').mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f"{drive_output / 'checkpoints'}/", f"{local / 'checkpoints'}/"], check=True)
    if (drive_output / 'metrics.jsonl').is_file(): shutil.copy2(drive_output / 'metrics.jsonl', local / 'metrics.jsonl')
    subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--config', str(write_pilot_config(arm)), '--resume', 'latest'], check=True)
    preserve_metrics(arm)

## Pilot A: peak learning rate 1e-5, updates 0–150

In [ ]:
run_fresh_pilot('lr_1e5')

## Pilot B: peak learning rate 2e-5, updates 0–150

In [ ]:
run_fresh_pilot('lr_2e5')

## Resume one interrupted pilot

In [ ]:
ARM_TO_RESUME = 'lr_1e5'
resume_pilot(ARM_TO_RESUME)

## Validation-only pilot comparison

In [ ]:
import math
rows = []
for arm, learning_rate in PILOT_LRS.items():
    root = DRIVE_ROOT / 'pilots' / arm / 'checkpoints'
    preferred = root / 'best_preferred.json'
    eligible = root / 'best_eligible.json'
    pointer_path = preferred if preferred.is_file() else eligible
    assert pointer_path.is_file(), f'{arm} has no retention-eligible candidate.'
    pointer = json.loads(pointer_path.read_text())
    checkpoint = root / pointer['checkpoint']
    verify_checkpoint(checkpoint)
    state = json.loads((checkpoint / 'trainer_state.json').read_text())
    rows.append({'arm': arm, 'learning_rate': learning_rate, 'pointer': pointer_path.stem, 'checkpoint': checkpoint.name, 'update': state['update'], 'sft_loss': state['best_preferred_sft_loss'] if pointer_path == preferred else state['best_eligible_sft_loss'], 'medical_degradation': math.exp(state['latest_medical_validation_loss'] - state['medical_validation_baseline_loss']) - 1, 'general_degradation': math.exp(state['latest_general_validation_loss'] - state['general_validation_baseline_loss']) - 1})
preferred_rows = [row for row in rows if row['pointer'] == 'best_preferred']
selected = min(preferred_rows or rows, key=lambda row: row['sft_loss'])
selection = {'selection_uses_test_data': False, 'selection_rule': 'lowest response-only SFT validation loss inside preferred dual-retention band', 'selected_arm': selected['arm'], 'selected_learning_rate': selected['learning_rate'], 'selected_checkpoint': selected['checkpoint'], 'pilots': rows}
selection_path = DRIVE_ROOT / 'pilot_selection.json'
selection_path.write_text(json.dumps(selection, indent=2, sort_keys=True) + '\n')
print(json.dumps(selection, indent=2))

# Fresh full Stage C training — standalone
Run this cell alone on a fresh GPU runtime after pilot selection.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, yaml
drive.mount('/content/drive')
repo = Path('/content/medical-slm'); archive = Path('/content/drive/MyDrive/medical-slm/stage-c-sft-data.tar'); checksum = Path(str(archive) + '.sha256')
url = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'; branch = 'main'
if repo.exists() and not (repo / '.git').is_dir(): shutil.rmtree(repo)
if not (repo / '.git').is_dir(): subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', url, str(repo)], check=True)
else: subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repo); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
sys.path.insert(0, str(repo / 'src')); importlib.invalidate_caches()
digest = hashlib.sha256(archive.read_bytes()).hexdigest(); assert digest == checksum.read_text().split()[0]
if not Path('datasets/tokenized/sft_stage_c_v1/manifest.json').is_file(): subprocess.run(['tar', '-xf', str(archive), '-C', str(repo)], check=True)
drive_root = Path('/content/drive/MyDrive/medical-slm-runs/stage_c_sft_v1'); parent_matches = sorted(Path('/content/drive/MyDrive/medical-slm-runs/stage_b_v2').rglob('checkpoint_00008000/checkpoint_manifest.json')); assert parent_matches
local_parent = Path('/content/stage_b_v2_parent/checkpoint_00008000'); local_parent.parent.mkdir(parents=True, exist_ok=True); subprocess.run(['rsync', '-a', f'{parent_matches[0].parent}/', f'{local_parent}/'], check=True)
selection = json.loads((drive_root / 'pilot_selection.json').read_text()); local_output = Path('/content/stage_c_full'); drive_output = drive_root / 'full'
assert not (drive_output / 'checkpoints/latest.json').exists(), 'Full run exists; use resume.'
if local_output.exists(): shutil.rmtree(local_output)
values = yaml.safe_load(Path('configs/training_stage_c_sft.yaml').read_text()); lr = selection['selected_learning_rate']; values.update({'output_directory': str(local_output), 'checkpoint_backup_directory': str(drive_output / 'checkpoints'), 'parent_checkpoint_directory': str(local_parent), 'precision': 'auto', 'learning_rate': lr, 'final_learning_rate': lr / 10, 'max_updates': 588})
config = Path('/content/training_stage_c_full.yaml'); config.write_text(yaml.safe_dump(values, sort_keys=False))
subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--config', str(config)], check=True)

# Resume full Stage C training — standalone
Run this cell alone after reconnecting to a GPU runtime.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, yaml
drive.mount('/content/drive')
repo = Path('/content/medical-slm'); archive = Path('/content/drive/MyDrive/medical-slm/stage-c-sft-data.tar'); checksum = Path(str(archive) + '.sha256')
url = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'; branch = 'main'
if repo.exists() and not (repo / '.git').is_dir(): shutil.rmtree(repo)
if not (repo / '.git').is_dir(): subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', url, str(repo)], check=True)
else: subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repo); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
sys.path.insert(0, str(repo / 'src')); importlib.invalidate_caches()
digest = hashlib.sha256(archive.read_bytes()).hexdigest(); assert digest == checksum.read_text().split()[0]
if not Path('datasets/tokenized/sft_stage_c_v1/manifest.json').is_file(): subprocess.run(['tar', '-xf', str(archive), '-C', str(repo)], check=True)
drive_root = Path('/content/drive/MyDrive/medical-slm-runs/stage_c_sft_v1'); parent_matches = sorted(Path('/content/drive/MyDrive/medical-slm-runs/stage_b_v2').rglob('checkpoint_00008000/checkpoint_manifest.json')); assert parent_matches
local_parent = Path('/content/stage_b_v2_parent/checkpoint_00008000'); local_parent.parent.mkdir(parents=True, exist_ok=True); subprocess.run(['rsync', '-a', f'{parent_matches[0].parent}/', f'{local_parent}/'], check=True)
selection = json.loads((drive_root / 'pilot_selection.json').read_text()); local_output = Path('/content/stage_c_full'); drive_output = drive_root / 'full'; local_checkpoints = local_output / 'checkpoints'; local_checkpoints.mkdir(parents=True, exist_ok=True)
assert (drive_output / 'checkpoints/latest.json').is_file(), 'No full checkpoint to resume.'
subprocess.run(['rsync', '-a', f"{drive_output / 'checkpoints'}/", f'{local_checkpoints}/'], check=True)
drive_metrics = drive_output / 'metrics.jsonl'
if drive_metrics.is_file(): shutil.copy2(drive_metrics, local_output / 'metrics.jsonl')
values = yaml.safe_load(Path('configs/training_stage_c_sft.yaml').read_text()); lr = selection['selected_learning_rate']; values.update({'output_directory': str(local_output), 'checkpoint_backup_directory': str(drive_output / 'checkpoints'), 'parent_checkpoint_directory': str(local_parent), 'precision': 'auto', 'learning_rate': lr, 'final_learning_rate': lr / 10, 'max_updates': 588})
config = Path('/content/training_stage_c_full.yaml'); config.write_text(yaml.safe_dump(values, sort_keys=False))
subprocess.run(['python', 'scripts/training/train_stage_c_sft.py', '--config', str(config), '--resume', 'latest'], check=True)

## Verify full-run completion or documented safety stop

In [ ]:
checkpoint_root = Path('/content/drive/MyDrive/medical-slm-runs/stage_c_sft_v1/full/checkpoints')
latest = json.loads((checkpoint_root / 'latest.json').read_text())
checkpoint = checkpoint_root / latest['checkpoint']
verify_checkpoint(checkpoint)
state = json.loads((checkpoint / 'trainer_state.json').read_text())
completed = (checkpoint_root / 'final_stage_c_sft.json').is_file()
safety_stop = state['consecutive_emergency_retention_breaches'] >= 2
early_stop = state['consecutive_sft_validation_non_improvements'] >= 3
assert completed or safety_stop or early_stop
assert state['non_finite_events'] == 0 and state['skipped_updates'] == 0
print({'latest': checkpoint.name, 'update': state['update'], 'epoch': state['epoch'], 'completed': completed, 'safety_stop': safety_stop, 'early_stop': early_stop, 'best_preferred_sft_loss': state['best_preferred_sft_loss']})

# Validation-only final checkpoint selection

This independently re-evaluates all unique checkpoint pointers. It refuses a hard-band fallback and never loads any test split. On a fresh runtime, rerun the bootstrap cell first.

In [ ]:
from pathlib import Path
import json, os, subprocess
REPOSITORY = Path('/content/medical-slm')
assert (REPOSITORY / '.git').is_dir(), 'Run the bootstrap cell first.'
os.chdir(REPOSITORY)
DRIVE_ROOT = Path('/content/drive/MyDrive/medical-slm-runs/stage_c_sft_v1')
CHECKPOINT_ROOT = DRIVE_ROOT / 'full/checkpoints'
BASELINE_REPORT = DRIVE_ROOT / 'stage_c_baseline.json'
VALIDATION_REPORT = DRIVE_ROOT / 'stage_c_candidate_validation.json'
assert CHECKPOINT_ROOT.is_dir() and BASELINE_REPORT.is_file()
subprocess.run(['python', 'scripts/evaluation/select_stage_c_checkpoint.py', '--checkpoint-root', str(CHECKPOINT_ROOT), '--baseline-report', str(BASELINE_REPORT), '--output', str(VALIDATION_REPORT)], check=True)
report = json.loads(VALIDATION_REPORT.read_text())
assert report['selection_uses_test_data'] is False
assert report['selected_candidate']['preferred'] is True
print({'selected_checkpoint': report['selected_checkpoint'], 'sft_loss': report['selected_candidate']['sft_validation']['loss'], 'medical_degradation': report['selected_candidate']['medical_perplexity_degradation_fraction'], 'general_degradation': report['selected_candidate']['general_perplexity_degradation_fraction']})
print('STAGE C VALIDATION-ONLY SELECTION: VERIFIED')

# Per-source validation: balanced versus specialist

Compare checkpoints 125 and 588 on all seven SFT validation sources before registering the specialist profile. This cell does not access test data.

In [ ]:
SOURCE_REPORT = DRIVE_ROOT / 'stage_c_source_validation.json'
subprocess.run(['python', 'scripts/evaluation/analyze_stage_c_sources.py', '--checkpoint-root', str(CHECKPOINT_ROOT), '--balanced-checkpoint', 'checkpoint_00000125', '--specialist-checkpoint', 'checkpoint_00000588', '--output', str(SOURCE_REPORT)], check=True)
source_report = json.loads(SOURCE_REPORT.read_text())
assert source_report['analysis_split'] == 'validation'
assert source_report['analysis_uses_test_data'] is False
for source, result in source_report['comparison_by_source'].items():
    print(source, {'balanced_loss': round(result['balanced_loss'], 6), 'specialist_loss': round(result['specialist_loss'], 6), 'perplexity_change_fraction': round(result['perplexity_change_fraction'], 6), 'accuracy_change': round(result['response_token_accuracy_change'], 6), 'improved': result['specialist_improves_loss']})
print(source_report['summary'])
print('STAGE C PER-SOURCE VALIDATION ANALYSIS: VERIFIED')

# Lock balanced and specialist profiles before test access

This immutable registration fixes checkpoint 588 as the primary medical-instruction specialist and checkpoint 125 as the balanced comparator. Run it exactly once before uploading or extracting sealed test data.

In [ ]:
PROFILE_REGISTRATION = DRIVE_ROOT / 'stage_c_profile_registration.json'
assert not PROFILE_REGISTRATION.exists(), f'Registration already exists: {PROFILE_REGISTRATION}'
subprocess.run(['python', 'scripts/evaluation/register_stage_c_profiles.py', '--candidate-validation', str(VALIDATION_REPORT), '--source-validation', str(SOURCE_REPORT), '--output', str(PROFILE_REGISTRATION)], check=True)
registration = json.loads(PROFILE_REGISTRATION.read_text())
assert registration['status'] == 'locked_before_test'
assert registration['primary_profile'] == 'medical_instruction_specialist'
assert registration['profiles']['medical_instruction_specialist']['checkpoint'] == 'checkpoint_00000588'
assert registration['profiles']['balanced_retention']['checkpoint'] == 'checkpoint_00000125'
print('STAGE C PROFILE REGISTRATION: VERIFIED AND LOCKED')

# One-time sealed test evaluation

First upload `stage-c-sealed-test.tar` and `stage-c-sealed-test.tar.sha256` to `MyDrive/medical-slm/`. This cell writes a started sentinel before test loading and permanently refuses an unnoticed rerun. Test results report the two pre-registered profiles; they cannot change their roles.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess
SEALED_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-c-sealed-test.tar')
SEALED_CHECKSUM = Path(str(SEALED_ARCHIVE) + '.sha256')
TEST_REPORT = DRIVE_ROOT / 'stage_c_test_evaluation.json'
TEST_SENTINEL = DRIVE_ROOT / 'stage_c_test_evaluation_status.json'
assert PROFILE_REGISTRATION.is_file(), 'Profiles must be locked before test access.'
assert SEALED_ARCHIVE.is_file() and SEALED_CHECKSUM.is_file()
assert not TEST_REPORT.exists() and not TEST_SENTINEL.exists(), 'A test attempt is already recorded.'
digest = hashlib.sha256()
with SEALED_ARCHIVE.open('rb') as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
expected = SEALED_CHECKSUM.read_text().split()[0]
assert digest.hexdigest() == expected
os.chdir(REPOSITORY)
subprocess.run(['tar', '-xf', str(SEALED_ARCHIVE), '-C', str(REPOSITORY)], check=True)
subprocess.run(['python', 'scripts/evaluation/evaluate_stage_c_test.py', '--checkpoint-root', str(CHECKPOINT_ROOT), '--profile-registration', str(PROFILE_REGISTRATION), '--output', str(TEST_REPORT), '--sentinel', str(TEST_SENTINEL)], check=True)
test_report = json.loads(TEST_REPORT.read_text())
sentinel = json.loads(TEST_SENTINEL.read_text())
assert sentinel['status'] == 'completed' and test_report['test_evaluated_once'] is True
print({'primary_profile': test_report['registered_primary_profile'], 'balanced_sft_test': test_report['balanced_retention']['sft_test'], 'specialist_sft_test': test_report['medical_instruction_specialist']['sft_test'], 'comparison': test_report['specialist_vs_balanced']})
print('STAGE C SEALED TEST EVALUATION: VERIFIED ONCE')